In [10]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *
from locallib.box import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [12]:
customer_name = 'Cadent'

In [13]:
data = Query(query = f"SELECT * FROM KPI_PeakSATLocation WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}')").execute(KPIHub_Conn)
box_file_id = data.iloc[0]['BoxFileId']
customer_id = data.iloc[0]['CustomerId']

In [14]:
box_obj = BoxFile(local_path='temp.xlsx', box_file_id =box_file_id)
box_obj.download()
df = pd.read_excel('temp.xlsx')
df['CustomerId'] = customer_id
df['ReportYear'] = df['Date'].dt.year
df['LastUpdated'] = datetime.now()
df.rename(columns = {'Region': 'BoundaryRegion'}, inplace = True)
box_obj.delete()    
KPI_PeakAboveSAT.update_table(arguments = {'DataFrame': df, 'PrimaryKey': 'PeakId', 'db_path': DB_PATH})

/home/sandbox/personal-repos/packages/locallib/box/BoxFile.py:17: UserWarning: The file temp.xlsx does not exist locally
  warnings.warn(f'The file {self.local_path.as_posix()} does not exist locally')


In [15]:
Query(query = f"SELECT * FROM KPI_PeakAboveSAT").execute(KPIHub_Conn)

,CustomerId,PeakName,PeakId,Date,ReportYear,WeekNumber,Disposition,LocalTime,BoundaryName,BoundaryRegion,EmissionRate,PeakGpsLatitude,PeakGpsLongitude,Easting,Northing,UserName,SurveyorUnit,AnalyzerSerialNumber,Hyperlink,LastUpdated
0,BD4D080B-1D12-D329-ABD0-39FEB9804E98,PC37E46,C37E46B1-0713-4991-BF04-EF3B5CDE31EB,2026-07-07 00:00:00,2026,28,1,04:27:19,B26 - 9607 - Warrington - St. Helens - Haydock #1,North West,24.702737,53.472711,-2.659475,356323.729863,397533.583858,Glynn.Evans@cadentgas.com,Cadent Surveyor #06 - LM75XVG,RFADS2357,https://pcubed2.eu.picarro.com/Live/Survey/494...,2026-07-08 03:52:15.559820
1,BD4D080B-1D12-D329-ABD0-39FEB9804E98,PC189DB,C189DBFD-B7E9-4C53-8DB4-16FE2F543074,2026-07-07 00:00:00,2026,28,1,04:14:39,B26 - 9607 - Warrington - St. Helens - Haydock #1,North West,26.459576,53.472538,-2.686499,354529.748632,397531.165172,Glynn.Evans@cadentgas.com,Cadent Surveyor #06 - LM75XVG,RFADS2357,https://pcubed2.eu.picarro.com/Live/Survey/632...,2026-07-08 03:52:15.559820
2,BD4D080B-1D12-D329-ABD0-39FEB9804E98,P4D7E1D,4D7E1DBD-0528-44AC-A1B5-657D80161F4B,2026-07-07 00:00:00,2026,28,1,03:16:38,B26 - 5277 - Fulham - Hounslow - Cranford,North London,23.546949,51.478025,-0.404309,510911.222084,176658.830713,Khushal.Hirani@cadentgas.com,Cadent Surveyor #10 - LL75ZZF,RFADS2367,https://pcubed2.eu.picarro.com/Live/Survey/b83...,2026-07-08 03:52:15.559820
3,BD4D080B-1D12-D329-ABD0-39FEB9804E98,P52F235,52F23558-5934-43D8-8298-696EAF452BDF,2026-07-07 00:00:00,2026,28,1,01:51:40,B26 - 5277 - Fulham - Hounslow - Cranford,North London,72.806306,51.478010,-0.404402,510904.776637,176657.041129,Khushal.Hirani@cadentgas.com,Cadent Surveyor #10 - LL75ZZF,RFADS2367,https://pcubed2.eu.picarro.com/Live/Survey/139...,2026-07-08 03:52:15.559820
4,BD4D080B-1D12-D329-ABD0-39FEB9804E98,P7A0125,7A01254C-682C-401D-B6FF-822B5D80DD94,2026-07-06 00:00:00,2026,28,1,23:54:03,B26 - 5277 - Fulham - Hounslow - Cranford,North London,42.670403,51.477828,-0.404675,510886.301275,176636.304432,Khushal.Hirani@cadentgas.com,Cadent Surveyor #10 - LL75ZZF,RFADS2367,https://pcubed2.eu.picarro.com/Live/Survey/139...,2026-07-08 03:52:15.559820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
800,BD4D080B-1D12-D329-ABD0-39FEB9804E98,P949807,949807EC-5BE2-489E-8BCC-6CEF9171D217,2026-01-27 00:00:00,2026,5,2,04:03:31,B25R2 - Stockport - Stockport - Bramhall North #1,North West,26.964661,53.385922,-2.176427,388362.784017,387689.952420,michaelgreenough@cadentgas.com,Cadent Surveyor #13 - LM75XBK,RFADS2371,https://pcubed2.eu.picarro.com/Live/Survey/cba...,2026-07-08 03:52:15.559820
801,BD4D080B-1D12-D329-ABD0-39FEB9804E98,P983813,98381321-744F-4CAB-B800-A815E8B3F81A,2026-01-27 00:00:00,2026,5,2,04:03:24,B25R2 - Stockport - Stockport - Bramhall North #1,North West,23.120044,53.386288,-2.176045,388388.299752,387730.596981,michaelgreenough@cadentgas.com,Cadent Surveyor #13 - LM75XBK,RFADS2371,https://pcubed2.eu.picarro.com/Live/Survey/cba...,2026-07-08 03:52:15.559820
802,BD4D080B-1D12-D329-ABD0-39FEB9804E98,P0575F8,0575F8A3-1708-4DCA-8E4A-9B1B3E3F512E,2026-01-11 00:00:00,2026,2,1,21:17:33,B25 - Rayleigh - Rochford - Hullbridge,North London,35.051292,51.616920,0.612224,580944.982702,194131.356162,fatih.gun@cadentgas.com,Cadent Surveyor #01 - LO75ZDT,RFADS2176,https://pcubed2.eu.picarro.com/Live/Survey/773...,2026-07-08 03:52:15.559820
803,BD4D080B-1D12-D329-ABD0-39FEB9804E98,PB12432,B124328F-3C6C-4507-8A8C-1109009736C3,2026-01-09 00:00:00,2026,2,1,00:14:27,B25 - Rayleigh - Rochford - Hullbridge,North London,23.829567,51.612550,0.608504,580704.807834,193636.177083,fatih.gun@cadentgas.com,Cadent Surveyor #01 - LO75ZDT,RFADS2176,https://pcubed2.eu.picarro.com/Live/Survey/635...,2026-07-08 03:52:15.559820
